# Stripe Schema Demo

This notebook demonstrates how to load a Stripe schema and environment config, and use them with the JsonAI library. It showcases tool chaining and output generation using Ollama.

You can switch between different schemas, environments, and Ollama models below.

In [ ]:
# Configuration: Choose schema, environment, and Ollama model
schemas = [
    "transfer_reversals_metadata",
    "tax_rates_metadata",
    "transfer_reversals"
]
environments = ["dev", "qa", "cte", "perf", "prod"]
ollama_model = "mistral:latest"  # Change as needed

# Select one schema and environment for demo
schema_choice = "transfer_reversals_metadata"  # or any from schemas
env = "dev"  # or any from environments

In [ ]:
from pathlib import Path
import json

base_dir = Path().cwd()
config_path = base_dir / "examples" / "stripe_schemas" / f"{schema_choice}.{env}.json"

# Load config
with open(config_path) as f:
    config = json.load(f)
    schema_path = base_dir / "examples" / "stripe_schemas" / config["schema"]

# Load schema
with open(schema_path) as f:
    schema = json.load(f)

print("Loaded config:", config)
print("Loaded schema:", schema)

In [ ]:
# Tool chaining setup
from jsonAI.tool_registry import ToolRegistry

def uppercase_value(value):
    return {"value": str(value).upper()}

def append_suffix(value, suffix="_demo"):
    return {"value": str(value) + suffix}

tool_registry = ToolRegistry()
tool_registry.register(uppercase_value)
tool_registry.register(append_suffix)

# Add tool chaining to schema (for demonstration)
schema["x-jsonai-tool-chain"] = [
    {
        "name": "uppercase_value",
        "arguments": {"value": "value"}
    },
    {
        "name": "append_suffix",
        "arguments": {"value": "value", "suffix": "_demo"}
    }
]

In [ ]:
# LLM-backed generation using Ollama
from jsonAI.model_backends import OllamaBackend
from jsonAI.main import Jsonformer, AsyncJsonformer
import asyncio

backend = OllamaBackend(ollama_model)
prompt = f"Generate a {schema_choice} object with sample values for {env}."
jsonformer = Jsonformer(
    model_backend=backend,
    json_schema=schema,
    prompt=prompt,
    tool_registry=tool_registry
)
async_jsonformer = AsyncJsonformer(jsonformer)

async def run_async():
    try:
        raw_data = jsonformer.generate_data()
        print(f"[DEBUG] Raw generated data: {raw_data}")
        result = await async_jsonformer()
        print(f"[DEBUG] Final result after tool chaining: {result}")
        return result
    except Exception as e:
        print(f"[ERROR] Exception: {e}")
        return None

result = asyncio.run(run_async())

In [ ]:
# Display the final result
import pprint
pprint.pprint(result)